# Point deduplicator for virma and lipas points

***
***NOTE*** <br>
Experimental code, not suitable for every dataset at this stage.
***

The objective of this deduplicator was to match points together between two spatial applications which both describe different outdoor sports and hiking places. We wanted to match the points from both applications that represented the same location in the real world. 

The deduplication process was done by first performing a K nearest neighbor search in QGIS based on the location of the points. From this search, the 3 nearest Lipas points within 200 meters were selected for each Virma point. This gave us a csv dataset from which we could determine whether any of the three closest points represented the same destination as the target Virma point.

From the three NN the nearest neighbor was decided based on the similarity of the points. Meaning that, in our dataset our points have a lot of metadata in addition to the location, such as name, category of the point, city etc. The points were considered the same if they had the same category or the exact same name since e.g., a beach could be "Uimaranta" in Lipas and "Uimapaikka" in Virma. For this we had a category mapping between our two applications the mapping.csv, which during this development was not quite complete yet but good enough. Borderline cases were decided by human. 

For those points that had so called "mismatched" categories, since they could be different in each application, we performed a similarity calculation based on the names, so that it could be easier to notice such cases where the names between the points were not exactly the same and the categories didn't match, but the points still represented the same destination. 

This application is all in all very simple and with this specific dataset a lot of manual work had to be put in so that all individual cases were spotted. In high level this works but some uniqueness may be overlooked.

Original dataset or the code output is not provided because of privacy reasons and the results are in virma_lipas_points_real.csv and the points that could be matched are in readable form in virma_lipas_points_matched.
***

In [ ]:
mapping_dict = dict(zip(category_mapping_df['class2_fi'], category_mapping_df['type_name']))


duplicates_df['mapped_lipas_category'] = duplicates_df['class2_fi'].map(mapping_dict)

duplicates_df['category_match'] = (duplicates_df['tyyppi_nimi_fi'] == duplicates_df['mapped_lipas_category']) | (duplicates_df['tyyppi_nimi_fi'] == duplicates_df['class2_fi']) | (duplicates_df['name_fi'] == duplicates_df['nimi_fi'])

mismatched_rows = duplicates_df[duplicates_df['category_match'] == False]

if not mismatched_rows.empty:
    print("Mismatched Categories:")
    display(mismatched_rows[["name_fi", "nimi_fi", 'class2_fi', 'tyyppi_nimi_fi', 'mapped_lipas_category']])
else:
    print("All categories match!")

In [ ]:
mismatched_rows[mismatched_rows["name_fi"] == "Rautilan rantasauna"]

In [ ]:
true_matches = duplicates_df[duplicates_df['category_match'] != False]
true_matches

In [ ]:
duplicated_ids2 = true_matches['gid'][true_matches['gid'].duplicated(keep=False)]

duplicates_df2 = true_matches[true_matches['gid'].isin(duplicated_ids2)]

duplicates_df2

In [ ]:
true_matches = true_matches[~((true_matches['name_fi'] == "Järvelän näköala- ja lintutorni") & (true_matches['nimi_fi'] == "Littoistenjärven lintutorni"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Kalikan uimaranta") & (true_matches['nimi_fi'] == "Turun Kalikan uimaranta"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Märynummen retkeilyreitin lähtöpiste") & (true_matches['nimi_fi'] == "Märynummen kuntoradan opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Märynummen retkeilyreitin lähtöpiste") & (true_matches['nimi_fi'] == "Märynummen retkeilyreitin opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Asmandia-laavu") & (true_matches['nimi_fi'] == "Asmandian grillikota"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Metsälehdon sauna ja Haimion vapaa sauna") & (true_matches['nimi_fi'] == "Haimion talviuintipaikka"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Littoistenjärven lintutorni") & (true_matches['nimi_fi'] == "Järvelän näköala- ja lintutorni"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Ellun polun opastuspiste") & (true_matches['nimi_fi'] == "Melassuon kuntoradan opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Märynummen retkeilyreitin opastuspiste") & (true_matches['nimi_fi'] == "Märynummen kuntoradan opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Palaisten uimaranta") & (true_matches['nimi_fi'] == "Palaisten talviuintipaikka"))]

In [ ]:
true_matches

In [ ]:
df = df.merge(duplicates_df, how='outer', indicator=True)
df = df[df['_merge'] == 'left_only']
df = df.drop(columns=['_merge'])
df

In [ ]:
needs_checking = mismatched_rows[mismatched_rows['mapped_lipas_category'].isna()]
needs_checking

In [ ]:
mismatched_rows = mismatched_rows.merge(needs_checking, how='outer', indicator=True)
mismatched_rows = mismatched_rows[mismatched_rows['_merge'] == 'left_only']
mismatched_rows = mismatched_rows.drop(columns=['_merge'])
mismatched_rows

In [ ]:
mismatched_rows[mismatched_rows['class2_fi'] == "Uimaranta"]

In [ ]:
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Särkijärven uimaranta ja rantasauna") & (mismatched_rows["nimi_fi"] == "Särkijärven uimapaikka")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Liipolan uimaranta") & (mismatched_rows["nimi_fi"] == "Liipolanjärven uimapaikka")]], ignore_index=True)
true_matches

In [ ]:
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["gid"] == 856) & (mismatched_rows["nimi_fi"] == "Mustajärven uimapaikka")]], ignore_index=True)
true_matches

In [ ]:
needs_checking[needs_checking["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

In [ ]:
mismatched_rows[mismatched_rows["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

In [ ]:
needs_checking = needs_checking[~needs_checking["gid"].isin(true_matches["gid"].unique())]
mismatched_rows = mismatched_rows[~mismatched_rows["gid"].isin(true_matches["gid"].unique())]

In [ ]:
not_matched = df[df["id_2"].isna()]
not_matched

In [ ]:
df = df.merge(not_matched, how='outer', indicator=True)
df = df[df['_merge'] == 'left_only']
df = df.drop(columns=['_merge'])
df

In [ ]:
df['mapped_lipas_category'] = df['class2_fi'].map(mapping_dict)

df['category_match'] = (df['tyyppi_nimi_fi'] == df['mapped_lipas_category']) | (df['tyyppi_nimi_fi'] == df['class2_fi']) | (df['name_fi'] == df['nimi_fi'])

mismatched_rows2 = df[df['category_match'] == False]

if not mismatched_rows2.empty:
    print("Mismatched Categories:")
    display(mismatched_rows2[["name_fi", "nimi_fi", 'class2_fi', 'tyyppi_nimi_fi', 'mapped_lipas_category']])
else:
    print("All categories match!")

In [ ]:
needs_checking = pd.concat([needs_checking, mismatched_rows2[mismatched_rows2['mapped_lipas_category'].isna()]], ignore_index=True)
needs_checking

In [ ]:
mismatched_rows2 = mismatched_rows2[~mismatched_rows2['mapped_lipas_category'].isna()]
mismatched_rows2

In [ ]:
true_matches = pd.concat([true_matches, df[df['category_match'] == True]], ignore_index=True)
true_matches

In [ ]:
mismatched_rows2[mismatched_rows2['class2_fi'] == "Uimaranta"]

In [ ]:
true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["name_fi"] == "Ölmosin uimaranta") & (mismatched_rows2["nimi_fi"] == "Västerbuktenin uimaranta")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["name_fi"] == "Bläsnäs") & (mismatched_rows2["nimi_fi"] == "Bläsnäsin uimala")]], ignore_index=True)
true_matches

In [ ]:
mismatched_rows2 = mismatched_rows2[~mismatched_rows2["gid"].isin(true_matches["gid"].unique())]

In [ ]:

true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["name_fi"] == "Maarian Mahdin ulkoilumaja") & (mismatched_rows2["nimi_fi"] == "Maarian mahti ry ulkoilumaja")]], ignore_index=True)
true_matches

In [ ]:
true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["gid"] == 302162) & (mismatched_rows2["nimi_fi"] == "Palmusen puiston frisbeegolfrata")]], ignore_index=True)
true_matches

In [ ]:
mismatched_rows2 = mismatched_rows2[~mismatched_rows2["gid"].isin(true_matches["gid"].unique())]

In [ ]:
mismatched_rows2[mismatched_rows2["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

In [ ]:
original_df = pd.read_csv("really_3_NN_virma_lipas.csv")

In [ ]:
from rapidfuzz import fuzz

for i, (name1, name2) in enumerate(zip(mismatched_rows2["name_fi"].tolist(), mismatched_rows2["nimi_fi"].tolist())):
    score = fuzz.token_sort_ratio(name1, name2)
    if score > 85:
        print(f"Match at index {i}: '{name1}' ~ '{name2}' (score: {score})")
    else:
        print(f"Different at index {i}: '{name1}' ≠ '{name2}' (score: {score})")

In [ ]:
original_df["tyyppi_nimi_fi"].value_counts()

In [ ]:
original_df["class2_fi"].value_counts()

SEURAAVAKSI: <br>
mismatched_rows2 chekattu ja oikeat matsit lisätty true_matches. Kaikki siellä olevat kohteet on mätsätty väärin eli täytyy laittaa niille, ettei mätsiä löytynyt ja lisätä not_matched. <br>
vielä jäljellä needs_checking ja mismatched rows

In [ ]:
needs_checking.shape

In [ ]:
mismatched_rows.shape

In [ ]:
import numpy as np

#mismatched_rows2[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan
#mismatched_rows2

In [ ]:
not_matched = pd.concat([not_matched, mismatched_rows2], axis=0, ignore_index=True)

In [ ]:
for i, (name1, name2) in enumerate(zip(mismatched_rows["name_fi"].tolist(), mismatched_rows["nimi_fi"].tolist())):
    score = fuzz.token_sort_ratio(name1, name2)
    if score > 85:
        print(f"Match at index {i}: '{name1}' ~ '{name2}' (score: {score})")
    else:
        print(f"Different at index {i}: '{name1}' ≠ '{name2}' (score: {score})")

'Valpperin hiihtokeskus ja urheilutalo' ≠ 'Valpperin hiihtokeskus' <br>
'Valpperin hiihtokeskus ja urheilutalo' ≠ 'Valpperin urheilutalo <br>
Auran kunnan Koivuniemen virkistysalue' ≠ 'Koivuniemen virkistysalue <br>
Högsåran Sandvikuddenin tulipaikka	...	Ruoanlaitto-/tulentekopaikka	Högsåran tulentekopaikka
Höglandin tulipaikka	...	Ruoanlaitto-/tulentekopaikka	Höglandin tulentekopaikka
Match at index 124: 'Taalintehdas DiscGolf Park' ~ 'Taalintehdas DiscGolfPark' (score: 98.0392156862745)




In [ ]:
for i, (name1, name2) in enumerate(zip(needs_checking["name_fi"].tolist(), needs_checking["nimi_fi"].tolist())):
    score = fuzz.token_sort_ratio(name1, name2)
    if score > 75:
        print(f"Match at index {i}: '{name1}' ~ '{name2}' (score: {score})")
    else:
        print(f"Different at index {i}: '{name1}' ≠ '{name2}' (score: {score})")

tru matches pitää poistaa hakkenpää

mis_matched_rows ja mis_matched_rows2 chekattu needs chenking tarvii vielä ja kerää kaikki kiikun kaakun olevat pisteet johonkin. 

In [ ]:
mismatched_rows[mismatched_rows["name_fi"] == "Valpperin hiihtokeskus ja urheilutalo"]

In [ ]:
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Auran kunnan Koivuniemen virkistysalue") & (mismatched_rows["nimi_fi"] == "Koivuniemen virkistysalue")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Taalintehdas DiscGolf Park") & (mismatched_rows["nimi_fi"] == "Taalintehdas DiscGolfPark")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Valpperin hiihtokeskus ja urheilutalo") & (mismatched_rows["nimi_fi"] == "Valpperin hiihtokeskus")]], ignore_index=True)

In [ ]:
true_matches[true_matches["name_fi"]== "Valpperin hiihtokeskus ja urheilutalo"]

In [ ]:
mismatched_rows[mismatched_rows["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

In [ ]:
mismatched_rows = mismatched_rows[~mismatched_rows["gid"].isin(true_matches["gid"].unique())]

In [ ]:
mismatched_rows = mismatched_rows.drop_duplicates(subset='gid', keep='first')

In [ ]:
#mismatched_rows[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan

In [ ]:
not_matched = pd.concat([not_matched, mismatched_rows], axis=0, ignore_index=True)

Himoisten rantasauna' ≠ 'Himoisten sauna ja uimaranta meillä kaksi eri pistettä, lippassa yksi<br>
- toteutus:
    - virman pisteet himoisten rantasauna ja himoisten uimaranta molemmat mätsättiin lippaan Himoisten sauna ja uimaranta pisteeseen

Rautilan rantasauna' ≠ 'Rautilan sauna ja uimapaikka meillä kaksi pistettä lippaassa yksi <br>
- toteutus:
    - virman pisteet rautilan rantasauna ja rautilan uimaranta molemmat mätsättiin lippaan Rautilan sauna ja uimapaikka pisteeseen

Särkijärven rantasauna	...	Uimapaikka	Särkijärven uimapaikka  -virmassa myös särkijärven uimaranta ja rantasauna <br>
- toteutus:
    - särkijärven rantasauna ei matchiä, koska lippaassa ei ole sitä, särkijärven uimaranta ja rantasauna mätsätty särkijärven uimapaikkaan

Aseman monitoimikenttä, jääkaukalo' ≠ 'Aseman kentän monitoimikaukalo' <br>
Aseman monitoimikenttä, jääkaukalo' ≠ 'Aseman koulun luistelukenttä <br>
- toteutus:
    - Aseman monitoimikenttä, jääkaukalo' = 'Aseman kentän monitoimikaukalo, koska more similar
    - Valpperin hiihtokeskus ja urheilutalo' ≠ 'Valpperin hiihtokeskus',  koska more similar <br>

In [ ]:
true_matches[true_matches["name_fi"] == "Valpperin hiihtokeskus ja urheilutalo"]

In [ ]:
true_matches.shape

In [ ]:
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Vaarniemen laavu ja näkötorni") & (needs_checking["nimi_fi"] == "Vaarniemen näkötorni")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Amospuiston kuntoportaat") & (needs_checking["nimi_fi"] == "Kuntoportaat")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Västanfjärds DiscGolfPark") & (needs_checking["nimi_fi"] == "Västanfjärd DiscGolfPark")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Kuntoportaat") & (needs_checking["nimi_fi"] == "Saukonojan kuntoportaat")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Virttaan Eräveikkojen maja") & (needs_checking["nimi_fi"] == "Virttaan kilpahiihtokeskus/ Erä-Veikkojen maja")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Kavilannummen mottorirata") & (needs_checking["nimi_fi"] == "Kavilannummen motocross-rata")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Agilityrata") & (needs_checking["nimi_fi"] == "MynSK:n koulutuskenttä")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Stora Masugnsträsketin kanoottilaituri ja vuokrakanootit") & (needs_checking["nimi_fi"] == "Stora Masugnsträsketin kanoottilaituri")]], ignore_index=True)

In [ ]:
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Rautilan rantasauna") & (needs_checking["nimi_fi"] == "Rautilan sauna ja uimapaikka")]], ignore_index=True)

In [ ]:
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Himoisten rantasauna") & (needs_checking["nimi_fi"] == "Himoisten sauna ja uimaranta")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Aseman monitoimikenttä, jääkaukalo") & (needs_checking["nimi_fi"] == "Aseman kentän monitoimikaukalo")]], ignore_index=True)

In [ ]:
true_matches.shape

In [ ]:
needs_checking[needs_checking["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

In [ ]:
needs_checking.shape

In [ ]:
needs_checking = needs_checking[~needs_checking["gid"].isin(true_matches["gid"].unique())]

In [ ]:
needs_checking = needs_checking.drop_duplicates(subset='gid', keep='first')

In [ ]:
#needs_checking[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan

In [ ]:
not_matched = pd.concat([not_matched, needs_checking], axis=0, ignore_index=True)

In [ ]:
pd.set_option("display.max_columns", None)

not_matched[not_matched.duplicated(subset='name_fi', keep=False)]

In [ ]:
df1_gids = set(true_matches["gid"])
df2_gids = set(not_matched["gid"])
df3_gids = set(original_df["gid"])

missing_from_1_and_2 = df3_gids - (df1_gids | df2_gids)

original_df[original_df["gid"].isin(missing_from_1_and_2)]

In [ ]:
##not_matched = not_matched.drop_duplicates(subset=["name_fi", "x_eureffin", "y_eureffin"], keep="first")

In [ ]:
not_matched.shape

In [ ]:
true_matches.shape

In [ ]:


not_matched[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan
not_matched.head()

In [ ]:
df1_gids = set(true_matches["gid"])
df2_gids = set(not_matched["gid"])
df3_gids = set(original_df["gid"])

missing_from_1_and_2 = df3_gids - (df1_gids | df2_gids)

original_df[original_df["gid"].isin(missing_from_1_and_2)].shape

Väärät mätsit true_matchessä <br>
Poistettu ja lisätty not matched <br>

Vuohensaaren luontopolun lähtöpiste		Vuohensaaren luontopolun opastuspiste <br>
Hakkenpää	Hakkenpää  <br>
Yxskärin kiinnittymispaikka 3	Retki- tai luonnonsatama	Yxskärin retkisatama	<br>
Yxskärin kiinnittymispaikka 2	Retki- tai luonnonsatama	Yxskärin retkisatama	<br>
Höglandin kiinnityslenkki	Retki- tai luonnonsatama	Höglandin retkisatama <br>
Höglandin kiinnityslenkki 2	Retki- tai luonnonsatama	Höglandin retkisatama <br>
Ekniemen lomakylä	Majoituspalvelu	Ekniemen lomakylä <br>
Lakjärven laavu 1	Yöpymislaavu tai -kota	Lakjärven laavu <br>
Matildajärventien pään opastaulu	Opastuspiste	Teijon luontotalo	Opastuspiste	Opastuspiste <br>
238	Matildajärven pysäköintialueen opastaulu	Opastuspiste	Teijon luontotalo	<br>


In [ ]:
true_matches[["name_fi", "class2_fi", "nimi_fi", "tyyppi_nimi_fi", "mapped_lipas_category"]]

In [ ]:
true_matches.shape

In [ ]:
not_matched.shape

In [ ]:
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Vuohensaaren luontopolun lähtöpiste") & (true_matches["nimi_fi"] == "Vuohensaaren luontopolun opastuspiste")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Hakkenpää") & (true_matches["nimi_fi"] == "Hakkenpää")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Yxskärin kiinnittymispaikka 3") & (true_matches["nimi_fi"] == "Yxskärin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Yxskärin kiinnittymispaikka 2") & (true_matches["nimi_fi"] == "Yxskärin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Höglandin kiinnityslenkki") & (true_matches["nimi_fi"] == "Höglandin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Höglandin kiinnityslenkki 2") & (true_matches["nimi_fi"] == "Höglandin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Lakjärven laavu 1") & (true_matches["nimi_fi"] == "Lakjärven laavu")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Matildajärventien pään opastaulu") & (true_matches["nimi_fi"] == "Teijon luontotalo")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Matildajärven pysäköintialueen opastaulu") & (true_matches["nimi_fi"] == "Teijon luontotalo")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Ekniemen lomakylä") & (true_matches["nimi_fi"] == "Ekniemen lomakylä") & (true_matches["class2_fi"] == "Majoituspalvelu")]], ignore_index=True)

In [ ]:
not_matched.shape

In [ ]:
true_matches[true_matches["gid"].isin(not_matched["gid"].unique())]["gid"].unique()

In [ ]:
true_matches = true_matches[~true_matches["gid"].isin(not_matched["gid"].unique())]

In [ ]:
true_matches.shape

In [ ]:
not_matched[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan
not_matched

ainoa kysymysmerkki enään: <br>
- hakkenpään uimaranta <br>
    - oli liian kaukana toisistaan, niin haku ei bongannu sitä onko edes samat pisteet
    - täytyykö lisätä manuaalisesti
    - poistetaanko virma duplicaatit


In [ ]:
# duplikaattienpoisto, jos nii ja koordinaatit mätsäävät

##not_matched = not_matched.drop_duplicates(subset=["name_fi", "x_eureffin", "y_eureffin"], keep="first")

In [ ]:
not_matched[not_matched["name_fi"] == "Hakkeenpään ranta"]

In [ ]:
true_matches.shape

In [ ]:
true_matches = pd.concat([true_matches, not_matched[(not_matched["name_fi"] == "Hakkeenpään ranta")]], ignore_index=True)
true_matches.shape

In [ ]:
not_matched.shape

In [ ]:
not_matched = not_matched[~not_matched["gid"].isin(true_matches["gid"].unique())]
not_matched.shape

In [ ]:
true_matches.loc[true_matches["gid"] == 1095, ["id_2", "nimi_fi", "tyyppi_nimi_fi"]] = [510287, "Hakkenpään uimapaikka", "Uimapaikka"]
true_matches[true_matches["name_fi"] == "Hakkeenpään ranta"]

In [ ]:
true_matches.loc[true_matches["class2_fi"].isin(["Uimapaikka", "Uimaranta"]),
    ["class2_fi", "tyyppi_nimi_fi", "name_fi", "nimi_fi"]]


In [ ]:
virma_lipas_mapping_df =  pd.concat([not_matched, true_matches], axis=0, ignore_index=True)
virma_lipas_mapping_df

In [ ]:
virma_lipas_mapping_df = virma_lipas_mapping_df[["id", "gid", "id_2"]].copy()
virma_lipas_mapping_df

In [ ]:
virma_lipas_mapping_df.rename(columns={"id_2": "mapped_lipas_id", "id": "virma_id"}, inplace=True)
virma_lipas_mapping_df

In [ ]:
virma_lipas_mapping_df[["virma_id", "mapped_lipas_id"]] = virma_lipas_mapping_df[["virma_id", "mapped_lipas_id"]].astype("Int64")

In [ ]:
virma_lipas_mapping_df.dtypes

In [ ]:
virma_lipas_mapping_df["mapped_lipas_id"] = virma_lipas_mapping_df["mapped_lipas_id"].fillna(0)
virma_lipas_mapping_df

In [ ]:
virma_lipas_mapping_df.dtypes

In [ ]:
virma_lipas_mapping_df.to_csv("virma_lipas_points_real.csv", index=False)

In [ ]:
dupes_all = not_matched[not_matched.duplicated(subset=["name_fi", "x_eureffin", "y_eureffin"], keep=False)]

In [ ]:
virma_lipas_points_matched = true_matches[["id", "gid", "id_2", "name_fi", "nimi_fi"]].copy()
virma_lipas_points_matched

In [ ]:
virma_lipas_points_matched.rename(columns={"id_2": "mapped_lipas_id", "id": "virma_id", "name_fi": "name_virma", "nimi_fi": "name_lipas"}, inplace=True)
virma_lipas_points_matched[["virma_id", "mapped_lipas_id"]] = virma_lipas_points_matched[["virma_id", "mapped_lipas_id"]].astype("Int64")
virma_lipas_points_matched

In [ ]:
virma_lipas_points_matched.dtypes

In [ ]:
virma_lipas_points_matched["lipas_link"] = virma_lipas_points_matched["mapped_lipas_id"].apply(
    lambda x: f"https://www.lipas.fi/liikuntapaikat/{x}" if pd.notna(x) else None
)
virma_lipas_points_matched

In [ ]:
virma_lipas_points_matched.to_csv("virma_lipas_points_matched.csv", index=False, na_rep="NULL")